## Notebook editing

`edit_notebook` is the production notebook mutation adapter. It keeps text transforms deterministic, applies notebook changes all-or-nothing, and returns structured diffs that MCP clients can inspect without parsing raw `.ipynb` JSON.

The shape is intentionally small: pure text helpers first, then one notebook/cell adapter. Read tools still provide context; this module only mutates notebooks.

In [ ]:
#| export
import copy
import difflib
import hashlib
import re
from pathlib import Path

from fastcore.nbio import mk_cell, read_nb
from fastcore.nbio import write_nb as _write_nb
from nbdev.doclinks import nbdev_export as _run_nb_export

from nbskill.foundation import (
    cell_source,
    clear_outputs,
    exported_py_path,
    find_cell_by_id,
    stamp_export_metadata,
    stamp_notebook_metadata,
    validate_code_cells,
)
from nbskill.parallel import notebook_locks
from nbskill.write import append_notebook_edit_feedback

_ExecutionApprovalRequired: nbskill: refusing to execute unapproved cell id=256e7289. Run it once yourself, or rerun nbskill with allow_new=True if you approve this source.

### Pure text helpers

The text helpers do not know about notebooks. They take strings, return strings, and raise clear `ValueError`s for invalid line bounds or malformed replacements. That makes cell-level and notebook-level editing share the same behavior.

In [ ]:
#| export
def _coerce_lines(lines):
    if lines is None: return []
    if isinstance(lines, str): return lines.splitlines()
    out = []
    for line in lines:
        parts = str(line).splitlines()
        out.extend(parts if parts else [""])
    return out


def _join_lines(lines):
    return chr(10).join(str(line) for line in lines)


def _line_no(value, n_lines, name):
    try: value = int(value)
    except (TypeError, ValueError) as err: raise ValueError(f"{name} must be an integer") from err
    if value < 0: value = n_lines + value + 1
    return value


def _norm_lines(text, start_line, end_line=None):
    """Return zero-based inclusive-exclusive indexes for 1-based line bounds."""
    lines = str(text).splitlines()
    n_lines = len(lines)
    if n_lines == 0: raise ValueError("cannot edit line ranges in an empty cell")
    start = _line_no(start_line, n_lines, "start_line")
    end = _line_no(start_line if end_line is None else end_line, n_lines, "end_line")
    if start < 1 or end < start or end > n_lines:
        raise ValueError(f"line bounds must be within 1:{n_lines}; got {start_line!r}:{end_line!r}")
    return start - 1, end


def _unified_diff(before, after, fromfile="before", tofile="after", context=2):
    """Return a compact unified diff, or an empty string when text is unchanged."""
    if before == after: return ""
    lines = difflib.unified_diff(
        str(before).splitlines(),
        str(after).splitlines(),
        fromfile=fromfile,
        tofile=tofile,
        lineterm="",
        n=context,
    )
    return chr(10).join(lines)


def insert_lines(text, insert_line, new_lines):
    """Insert `new_lines` before a 1-based line boundary in `text`; use n+1 to append."""
    lines = str(text).splitlines()
    try: idx = int(insert_line)
    except (TypeError, ValueError) as err: raise ValueError("insert_line must be an integer") from err
    if idx < 0: idx = len(lines) + idx + 2
    if idx < 1 or idx > len(lines) + 1: raise ValueError(f"insert_line must be within 1:{len(lines) + 1}; got {insert_line!r}")
    return _join_lines([*lines[:idx - 1], *_coerce_lines(new_lines), *lines[idx - 1:]])


def replace_lines(text, start_line, end_line=None, replacement_lines=None):
    """Replace a 1-based inclusive line range in `text`."""
    lines = str(text).splitlines()
    start, end = _norm_lines(text, start_line, end_line)
    return _join_lines([*lines[:start], *_coerce_lines(replacement_lines), *lines[end:]])


def delete_lines(text, start_line=None, end_line=None, re_filter=None, invert_filter=False):
    """Delete a 1-based inclusive line range, or lines matching `re_filter`."""
    lines = str(text).splitlines()
    if re_filter is not None:
        pattern = re.compile(re_filter)
        kept = []
        for line in lines:
            matched = bool(pattern.search(line))
            if invert_filter: matched = not matched
            if not matched: kept.append(line)
        return _join_lines(kept)
    if start_line is None: raise ValueError("delete_lines needs start_line unless re_filter is set")
    start, end = _norm_lines(text, start_line, end_line)
    return _join_lines([*lines[:start], *lines[end:]])


def replace_text(text, old, new, start_line=None, end_line=None):
    """Replace literal `old` with `new`, optionally inside a line range."""
    if old in {None, ""}: raise ValueError("replace_text needs a non-empty old value")
    source = str(text)
    if start_line is None and end_line is None: return source.replace(str(old), str(new))
    lines = source.splitlines()
    start, end = _norm_lines(source, start_line or 1, end_line)
    segment = _join_lines(lines[start:end]).replace(str(old), str(new))
    return _join_lines([*lines[:start], *segment.splitlines(), *lines[end:]])


def replace_texts(text, replacements=None, olds=None, news=None, start_line=None, end_line=None):
    """Apply several literal text replacements in order."""
    if replacements is None:
        if olds is None or news is None: raise ValueError("replace_texts needs replacements or olds/news")
        if len(olds) != len(news): raise ValueError("olds and news must have the same length")
        replacements = [{"old": old, "new": new} for old, new in zip(olds, news)]
    result = str(text)
    for item in replacements:
        result = replace_text(result, item.get("old"), item.get("new", ""), start_line=start_line, end_line=end_line)
    return result

The public helper functions `insert_lines`, `replace_lines`, `delete_lines`, `replace_text`, and `replace_texts` use one-based line bounds for `insert_line`, `start_line`, and `end_line`, plus the same text semantics that `edit_notebook` applies to notebook cells.

In [ ]:
sample = "alpha\nbeta\ngamma"
insert_lines(sample, 2, ["inserted"]), replace_lines(sample, 2, 2, ["BETA"]), delete_lines(sample, re_filter="beta"), replace_text(sample, "alpha", "ALPHA"), replace_texts(sample, [dict(old="alpha", new="A"), dict(old="gamma", new="G")])

In [ ]:
#| hide
assert _coerce_lines(["a", "", "b\n\nc"]) == ["a", "", "b", "", "c"]
assert insert_lines("a\nc", 2, ["b"]) == "a\nb\nc"
assert insert_lines("a\nb", 1, ["start"]) == "start\na\nb"
assert insert_lines("a\nb", 3, ["end"]) == "a\nb\nend"
assert replace_lines("a\nb\nc", 2, 2, ["B"]) == "a\nB\nc"
assert delete_lines("a\nb\nc", 2, 2) == "a\nc"
assert delete_lines("keep\ndrop", re_filter="drop") == "keep"
assert replace_text("one two one", "one", "1") == "1 two 1"
assert replace_texts("a b", [{"old": "a", "new": "A"}, {"old": "b", "new": "B"}]) == "A B"
assert _unified_diff("", _join_lines(["alpha", "", "beta"])).splitlines()[2:] == ["@@ -0,0 +1,3 @@", "+alpha", "+", "+beta"]
assert _unified_diff("same", "same") == ""

### Notebook adapter

The adapter selects cells, applies one deterministic operation per selected cell, validates the edited notebook before writing, clears stale outputs for changed code cells, stamps metadata, exports with nbdev, and returns structured results. The whole call succeeds or fails as one unit.

In [ ]:
#| export
_TEXT_OPS = {"replace_lines", "insert_lines", "delete_lines", "replace_text", "replace_texts"}
_STRUCTURAL_OPS = {"replace_cell", "insert_cells", "delete_cells", "move_cells"}
_EDIT_OPS = _TEXT_OPS | _STRUCTURAL_OPS


def _source_hash(source):
    return hashlib.sha256(str(source).encode("utf-8")).hexdigest()[:12]


def _notebook_hash(nb):
    payload = chr(30).join(f"{getattr(cell, 'id', '')}:{cell_source(cell)}" for cell in nb.cells)
    return _source_hash(payload)


def _path_inside_cwd(path):
    try: Path(path).expanduser().resolve(strict=False).relative_to(Path.cwd().resolve())
    except (OSError, ValueError): return False
    return True


def _cell_by_id_map(nb):
    return {getattr(cell, "id", None): (idx, cell) for idx, cell in enumerate(nb.cells)}


def _require_op(edit):
    if not isinstance(edit, dict): raise ValueError("each edit must be a dict")
    op = edit.get("op")
    if op not in _EDIT_OPS: raise ValueError(f"unsupported edit op {op!r}")
    return op


def _replacement_source(edit):
    if "source_lines" in edit: lines = edit["source_lines"]
    elif "source" in edit: lines = edit["source"]
    else: raise ValueError("replace_cell needs source_lines or source")
    source = _join_lines(_coerce_lines(lines))
    if source == "": raise ValueError("replace_cell cannot delete a cell; use delete_cells")
    return source


def _cell_from_source(source, cell_type="code"):
    cell = mk_cell(source, cell_type=cell_type)
    clear_outputs(cell)
    return cell


def _cells_from_structs(cells, default_cell_type="code"):
    if not cells: raise ValueError("insert_cells needs a non-empty cells list")
    made = []
    for spec in cells:
        spec = spec or {}
        cell_type = spec.get("cell_type", default_cell_type)
        source = _join_lines(_coerce_lines(spec.get("source_lines", spec.get("source", []))))
        made.append(_cell_from_source(source, cell_type=cell_type))
    return made


def _cell_content_match(cell, edit):
    if edit.get("cell_type") and getattr(cell, "cell_type", None) != edit.get("cell_type"): return False
    checks = []
    source = cell_source(cell)
    if edit.get("contains") is not None: checks.append(str(edit["contains"]) in source)
    if edit.get("re_filter") is not None: checks.append(bool(re.search(str(edit["re_filter"]), source, re.MULTILINE)))
    matched = all(checks) if checks else True
    return not matched if edit.get("invert_filter") else matched


def _selected_indices(nb, edit):
    ids = []
    if edit.get("cell_id") is not None: ids = [edit["cell_id"]]
    elif edit.get("cell_ids") is not None: ids = list(edit["cell_ids"])
    elif edit.get("target") == "all":
        return [idx for idx, cell in enumerate(nb.cells) if _cell_content_match(cell, edit)]
    else:
        raise ValueError(f"{edit.get('op')} needs cell_id, cell_ids, or target='all'")
    seen = _cell_by_id_map(nb)
    missing = [cell_id for cell_id in ids if cell_id not in seen]
    if missing: raise ValueError(f"unknown cell id(s): {missing}")
    return [seen[cell_id][0] for cell_id in ids if _cell_content_match(seen[cell_id][1], edit)]


def _check_expected_hash(nb, edit, indices):
    expected = edit.get("expected_hash")
    if expected is None: return
    if isinstance(expected, dict):
        for idx in indices:
            cell = nb.cells[idx]
            cell_id = getattr(cell, "id", "")
            want = expected.get(cell_id)
            if want is not None and want != _source_hash(cell_source(cell)):
                raise ValueError(f"expected_hash mismatch for cell {cell_id}")
        return
    if len(indices) != 1: raise ValueError("single expected_hash only works with one selected cell")
    cell = nb.cells[indices[0]]
    if str(expected) != _source_hash(cell_source(cell)):
        raise ValueError(f"expected_hash mismatch for cell {getattr(cell, 'id', '')}")


def _edit_cell_source(source, edit):
    op = edit["op"]
    if op == "replace_lines":
        return replace_lines(source, edit.get("start_line"), edit.get("end_line"), edit.get("replacement_lines", []))
    if op == "insert_lines":
        return insert_lines(source, edit.get("insert_line", edit.get("start_line")), edit.get("new_lines", edit.get("source_lines", [])))
    if op == "delete_lines":
        return delete_lines(source, edit.get("start_line"), edit.get("end_line"), edit.get("line_filter"), bool(edit.get("invert_filter")))
    if op == "replace_text":
        return replace_text(source, edit.get("old"), edit.get("new", ""), edit.get("start_line"), edit.get("end_line"))
    if op == "replace_texts":
        return replace_texts(source, edit.get("replacements"), edit.get("olds"), edit.get("news"), edit.get("start_line"), edit.get("end_line"))
    raise ValueError(f"{op!r} is not a text operation")


def _append_cell_diff(diffs, cell, before, after, op):
    diff = _unified_diff(before, after, fromfile=f"{getattr(cell, 'id', '')}:before", tofile=f"{getattr(cell, 'id', '')}:after")
    diffs.append({
        "op": op,
        "cell_id": getattr(cell, "id", ""),
        "cell_type": getattr(cell, "cell_type", ""),
        "changed": before != after,
        "matches": 0 if before == after else 1,
        "before_hash": _source_hash(before),
        "after_hash": _source_hash(after),
        "diff": diff,
    })


def _apply_text_edit(nb, edit, diffs, affected):
    indices = _selected_indices(nb, edit)
    _check_expected_hash(nb, edit, indices)
    for idx in indices:
        cell = nb.cells[idx]
        before = cell_source(cell)
        after = _edit_cell_source(before, edit)
        _append_cell_diff(diffs, cell, before, after, edit["op"])
        if after != before:
            cell.source = after
            clear_outputs(cell)
            affected.append(getattr(cell, "id", ""))


def _insert_at(nb, anchor_id, where, new_cells):
    idx, _ = find_cell_by_id(nb.cells, anchor_id)
    if where not in {"before", "after"}: raise ValueError("where must be 'before' or 'after'")
    target = idx if where == "before" else idx + 1
    for offset, cell in enumerate(new_cells): nb.cells.insert(target + offset, cell)
    return [getattr(cell, "id", "") for cell in new_cells]


def _apply_structural_edit(nb, edit, diffs, affected, default_cell_type):
    op = edit["op"]
    if op == "replace_cell":
        indices = _selected_indices(nb, edit)
        if len(indices) != 1: raise ValueError("replace_cell needs exactly one selected cell")
        _check_expected_hash(nb, edit, indices)
        cell = nb.cells[indices[0]]
        before = cell_source(cell)
        after = _replacement_source(edit)
        cell_type = edit.get("cell_type", getattr(cell, "cell_type", default_cell_type))
        new_cell = _cell_from_source(after, cell_type=cell_type)
        new_cell.id = getattr(cell, "id", new_cell.id)
        _append_cell_diff(diffs, cell, before, after, op)
        if after != before or getattr(cell, "cell_type", None) != cell_type:
            nb.cells[indices[0]] = new_cell
            affected.append(getattr(new_cell, "id", ""))
        return
    if op == "insert_cells":
        new_cells = _cells_from_structs(edit.get("cells"), default_cell_type=default_cell_type)
        inserted = _insert_at(nb, edit.get("anchor_id"), edit.get("where", "after"), new_cells)
        affected.extend(inserted)
        inserted_source = "\n---\n".join(cell_source(cell) for cell in new_cells)
        diffs.append({
            "op": op,
            "cell_id": "",
            "changed": True,
            "inserted_cell_ids": inserted,
            "before_hash": "",
            "after_hash": _source_hash(inserted_source),
            "diff": _unified_diff("", inserted_source, fromfile="insert:before", tofile="insert:after"),
        })
        return
    if op == "delete_cells":
        indices = _selected_indices(nb, edit)
        if not indices: raise ValueError("delete_cells matched no cells")
        _check_expected_hash(nb, edit, indices)
        for idx in sorted(indices, reverse=True):
            cell = nb.cells[idx]
            affected.append(getattr(cell, "id", ""))
            diffs.append({"op": op, "cell_id": getattr(cell, "id", ""), "changed": True, "before_hash": _source_hash(cell_source(cell)), "after_hash": "", "diff": _unified_diff(cell_source(cell), "")})
            del nb.cells[idx]
        return
    if op == "move_cells":
        indices = _selected_indices(nb, edit)
        if not indices: raise ValueError("move_cells matched no cells")
        before_order = [getattr(cell, "id", "") for cell in nb.cells]
        moving = [nb.cells[idx] for idx in indices]
        moving_ids = [getattr(cell, "id", "") for cell in moving]
        remaining = [cell for idx, cell in enumerate(nb.cells) if idx not in set(indices)]
        anchor_id = edit.get("anchor_id")
        where = edit.get("where", "after")
        anchor_positions = {getattr(cell, "id", None): idx for idx, cell in enumerate(remaining)}
        if anchor_id not in anchor_positions: raise ValueError(f"unknown anchor_id {anchor_id!r}")
        target = anchor_positions[anchor_id] + (1 if where == "after" else 0)
        nb.cells[:] = [*remaining[:target], *moving, *remaining[target:]]
        after_order = [getattr(cell, "id", "") for cell in nb.cells]
        affected.extend(moving_ids)
        diff = "\n".join([
            f"moved cells: {', '.join(moving_ids)}",
            f"anchor: {where} {anchor_id}",
            "before order: " + " -> ".join(before_order),
            "after order: " + " -> ".join(after_order),
        ])
        diffs.append({
            "op": op, "cell_id": "", "changed": True, "moved_cell_ids": moving_ids,
            "anchor_id": anchor_id, "where": where, "before_order": before_order,
            "after_order": after_order, "diff": diff,
        })
        return
    raise ValueError(f"unsupported structural op {op!r}")

In [ ]:
#| export
def edit_notebook(
    path,
    edits,
    validate_code=True,
    default_cell_type="code",
    auto_feedback=True,
    feedback_timeout=10,
    feedback_safe=True,
    detail="summary",
):
    """Apply structured notebook edits atomically and return MCP-friendly details."""
    if not edits: raise ValueError("edits must be a non-empty list")
    path = Path(path)
    normalized = [dict(edit) for edit in edits]
    for edit in normalized: _require_op(edit)

    with notebook_locks(path):
        nb = read_nb(path)
        before_hash = _notebook_hash(nb)
        trial = copy.deepcopy(nb)
        diffs, affected = [], []
        for edit in normalized:
            op = edit["op"]
            if op in _TEXT_OPS: _apply_text_edit(trial, edit, diffs, affected)
            else: _apply_structural_edit(trial, edit, diffs, affected, default_cell_type)
        affected = [cell_id for cell_id in dict.fromkeys(affected) if cell_id]
        changed = before_hash != _notebook_hash(trial)
        if validate_code and changed: validate_code_cells(trial.cells)
        exported = False
        if changed:
            for cell in trial.cells:
                if getattr(cell, "id", None) in set(affected): clear_outputs(cell)
            stamp_notebook_metadata(trial)
            _write_nb(trial, path)
            py_path = exported_py_path(path, trial)
            if py_path is not None and _path_inside_cwd(path):
                _run_nb_export(path=str(path))
                if py_path.exists():
                    stamp_export_metadata(trial, py_path)
                    _write_nb(trial, path)
                    exported = True
        after_hash = _notebook_hash(read_nb(path)) if changed else before_hash

    changed_diffs = [item for item in diffs if item.get("changed")]
    status = "changed" if changed else "no_change"
    text = f"edit_notebook {status}: {len(changed_diffs)} changed operation(s), {len(affected)} affected cell(s)"
    diff_text = "\n\n".join(item["diff"] for item in changed_diffs if item.get("diff"))
    if diff_text: text = f"{text}\n\n{diff_text}"
    if changed:
        text = append_notebook_edit_feedback(
            text, path, affected, auto_feedback=auto_feedback, feedback_timeout=feedback_timeout, feedback_safe=feedback_safe
        )
    return {
        "ok": True,
        "changed": changed,
        "no_change": not changed,
        "affected_cell_ids": affected,
        "diffs": diffs,
        "before_hash": before_hash,
        "after_hash": after_hash,
        "exported": exported,
        "warnings": [],
        "text": text,
    }

A notebook-wide rename is just a `replace_text` operation with `target="all"`. That path is the default for deterministic refactors such as renaming a public symbol across Markdown, examples, and tests.

In [ ]:
from nbskill.foundation import write_demo_notebook

with write_demo_notebook("edit_notebook_example.ipynb", cells=[mk_cell("old_name = 1")]) as example_path:
    result = edit_notebook(
        example_path,
        [dict(op="replace_text", target="all", old="old_name", new="new_name")],
        auto_feedback=False,
    )
    source = read_nb(example_path).cells[0].source
source

In [ ]:
#| hide
from contextlib import contextmanager
from fastcore.nbio import new_nb
from nbskill.foundation import write_demo_notebook


@contextmanager
def _outside_edit_probe():
    outside = Path("/private/tmp/nbskill_edit_outside_export_probe.ipynb")
    outside_py = Path("nbskill/_outside_edit_probe.py")
    outside.unlink(missing_ok=True)
    outside_py.unlink(missing_ok=True)
    outside_nb = new_nb([mk_cell("#| default_exp _outside_edit_probe"), mk_cell("probe = 1")])
    _write_nb(outside_nb, outside)
    try:
        yield outside, outside_py, outside_nb
    finally:
        outside.unlink(missing_ok=True)
        outside_py.unlink(missing_ok=True)

In [ ]:
#| hide
from nbskill.foundation import write_demo_notebook

with write_demo_notebook("edit_notebook_demo.ipynb", cells=[mk_cell("value = 1"), mk_cell("value")]) as demo:
    nb = read_nb(demo)
    cell_id = nb.cells[0].id
    assert callable(edit_notebook)
    result = edit_notebook(
        demo,
        [{"op": "replace_lines", "cell_id": cell_id, "start_line": 1, "end_line": 1, "replacement_lines": ["value = 2"]}],
        auto_feedback=False,
    )
    assert result["ok"] and result["changed"]
    assert "---" in result["text"]
    assert "exported with nbdev" not in result["text"]
    assert read_nb(demo).cells[0].source == "value = 2"
    bad_hash = [{"op": "replace_text", "cell_id": cell_id, "old": "2", "new": "3", "expected_hash": "nope"}]
    try:
        edit_notebook(demo, bad_hash, auto_feedback=False)
        raise AssertionError("hash mismatch should fail")
    except ValueError as err:
        assert "expected_hash" in str(err)
    assert read_nb(demo).cells[0].source == "value = 2"

with _outside_edit_probe() as (outside, outside_py, outside_nb):
    outside_result = edit_notebook(
        outside,
        [dict(op="replace_text", cell_id=outside_nb.cells[1].id, old="1", new="2")],
        auto_feedback=False,
    )
    assert outside_result["changed"] and not outside_result["exported"]
    assert not outside_py.exists()

In [ ]:
#| default_exp edit